# Interferometric X-Band Observations of the Sun

## AY 121 — Lab 03: Radio Interferometry

This notebook presents a complete analysis of interferometric X-band ($\sim 10.5\;\mathrm{GHz}$) observations of the Sun collected with the two-element east–west interferometer at New Campbell Hall (NCH), UC Berkeley. The analysis proceeds in three stages:

1. **Phenomenological modelling** — derive the interferometer response from first principles and build forward models for point sources and extended (solar-disk) emission.
2. **Baseline determination** — fit observed fringes to extract the east–west and north–south baseline components $(b_{\mathrm{ew}}, b_{\mathrm{ns}})$ using **five independent methods** (FFT, phase-slope, lag-spectrum, complex nonlinear least-squares, and a brute-force grid search).
3. **Solar science** — measure the angular diameter of the Sun from the Bessel-function modulation of fringe amplitude, characterise sunspot signatures from non-zero residual visibility at the Bessel nulls (both in *amplitude* and *phase*, the latter giving the EW offset of the spot from disk centre), and compare against the lidar-measured baseline and the SDO/HMI sunspot catalogue.

A baseline prior is available from a direct in-situ lidar measurement (iPhone 13 Pro) of the antenna phase-centre separation:

$$b_{\mathrm{ew}}^{\mathrm{lidar}} = 15.17 \pm 0.30\;\mathrm{m},\qquad b_{\mathrm{ns}}^{\mathrm{lidar}} = 1.36 \pm 0.30\;\mathrm{m}.$$

The 30 cm 1-sigma uncertainty is a deliberately conservative floor that combines the iPhone lidar's single-shot accuracy at ~15 m range with the unmodelled contribution from identifying the antenna phase centre by eye on a real dish. These values are stored in `utils/constants.py` as `NOMINAL_B_EW_M` / `NOMINAL_B_NS_M` and used as the prior for the nonlinear baseline fits and as the cross-check against the recovered values.

---

## Scientific roadmap

### What we want to know

This lab uses a two-element east–west radio interferometer to answer **three concrete questions** about the Sun and our own instrument:

1. **How big is the Sun at 10 GHz?** The optical photospheric diameter is $\theta_\odot^{\mathrm{opt}} \approx 31.6'$ (operational value from [AY121-Lab3]; physical mean $\sim 31.99'$ from [BCD98]), but the centimetre Sun is bigger because the radio emission comes from the chromosphere and low corona, a few thousand km above the photosphere ([Stix] §10.3, [Dulk85] §III). The closest *measured* published radio radius is the NoRH 17 GHz value of [Selhorst04]: $\theta_\odot(17\,\mathrm{GHz}) = 32.55 \pm 0.05'$ (mean over 3800+ NoRH maps spanning 1992–2003); 10 GHz is expected to be *slightly larger* (longer wavelength → higher chromosphere), so the comparison value for our observation is $\sim 32.6'$. Can we measure that $\sim 1\,\%$ excess directly from a single afternoon of fringes? *(Answered in notebook 05.)*
2. **Is there an active region on the visible solar disk today, and where is it?** A localised bright (or dark) feature on top of the disk produces a non-zero residual visibility at the Bessel nulls of the uniform-disk model — both an *amplitude* signature (flux fraction $f$) and a *phase* signature (EW offset $\Delta\alpha$). Cross-checking against the NOAA SWPC / SDO HMI active-region catalogue for the observation date converts a fringe wiggle into a piece of solar physics. *(Answered in notebook 05.)*
3. **What is our array geometry, and can the fringes themselves measure it?** We have a direct in-situ lidar measurement of the antenna phase-centre separation, $b_{\mathrm{ew}} = 15.17 \pm 0.30\;\mathrm{m}$ and $b_{\mathrm{ns}} = 1.36 \pm 0.30\;\mathrm{m}$. Can we recover those numbers from the fringe data alone (a meta-result about the *technique*)? *(Answered in notebook 04.)*

These three questions are answered by three independent measurements that share one calibrated dataset.

### How the notebooks fit together

```
   ┌────────────────────────────────┐
   │ 01  Theory                     │   (you are here)
   │     - geometric delay          │
   │     - fringe equation          │
   │     - VC–Z + Bessel envelope   │
   │     - 5 baseline methods       │
   │     - sunspot diagnostics      │
   └───────────────┬────────────────┘
                   │
   ┌───────────────▼────────────────┐
   │ 02a / 02b  Data inspection     │   "what is the signal,
   │     - raw V(ν, h)              │    what does our cleanup
   │     - chip-to-chip gain check  │    do to it?"
   │     - per-capture σ_V          │
   │     - adaptive DC correction   │
   │     - synthetic injection test │
   └───────────────┬────────────────┘
                   │
   ┌───────────────▼────────────────┐
   │ 03  Phenomenological models    │   "what does the lidar
   │     - predict fringe period    │    prior predict, and
   │     - predict Bessel envelope  │    where do we test it?"
   │     - lidar-prior overlay      │
   └───────────────┬────────────────┘
                   │
   ┌───────────────▼────────────────┐   ┌────────────────────────┐
   │ 04  Baseline determination     │   │ Question 3 answered:   │
   │     - 5 independent methods    │──▶│ array geometry from    │
   │     - lidar cross-check        │   │ fringes vs lidar       │
   │     - calibrated u-axis        │   └────────────────────────┘
   └───────────────┬────────────────┘
                   │ (calibration of the u-axis)
                   │
   ┌───────────────▼────────────────┐   ┌────────────────────────┐
   │ 05  Solar science              │   │ Questions 1 + 2:       │
   │     - Bessel envelope fit      │──▶│ radio diameter +       │
   │     - sunspot amplitude + phase│   │ active region (Δα, f)  │
   │     - SDO/HMI cross-check      │   └────────────────────────┘
   └────────────────────────────────┘
```

The right-hand column lists the *scientific question each notebook answers*. Notebooks 02–04 are setup; **the science is in notebook 05**, and the synthesis at the end of nb 05 ties all three results together.

---

## 1.1 The Adding Interferometer and Path Delay

Consider two antennas separated by a baseline vector $\mathbf{b}$ observing a distant point source. The electric field at antenna 1 is

$$E_1(t) = E_0 \cos(2\pi\nu\, t),$$

while at antenna 2 the signal arrives with a geometric time delay $\tau_g$:

$$E_2(t) = E_0 \cos\!\bigl[2\pi\nu\,(t + \tau_{\mathrm{tot}})\bigr],$$

where $\tau_{\mathrm{tot}} = \tau_g + \tau_c$ is the total delay, comprising the geometric delay $\tau_g$ (which depends on source position and baseline geometry) and a constant instrumental/cable delay $\tau_c$.

### Geometric delay for an EW + NS baseline

For an interferometer at geographic latitude $L$ with east–west baseline component $b_{\mathrm{ew}}$ and north–south component $b_{\mathrm{ns}}$, the geometric delay for a source at declination $\delta$ and hour angle $h$ is

$$\boxed{\tau_g(h) = \frac{b_{\mathrm{ew}}}{c}\cos\delta\,\sin h + \frac{b_{\mathrm{ns}}}{c}\sin L\,\cos\delta\,\cos h.}$$

(Standard rotation from the local topocentric $(E, N, U)$ frame to the equatorial $(X, Y, Z)$ frame; see [TMS3] eqs. 4.1–4.4 and the worked derivation in [TMS3] §4.1.)

**Sign convention.** $\tau_g > 0$ means the wavefront reaches antenna 1 before antenna 2; equivalently, $\tau_g = \mathbf{b}_{21}\cdot\hat{s}/c$ where $\mathbf{b}_{21}$ points *from* antenna 2 (east/north) *to* antenna 1 (west/south). In particular, when the source is rising ($h < 0$, east of the meridian), the wavefront reaches the east antenna first, giving $\tau_g < 0$.

**Derivation.** The baseline vector in the local topocentric frame is $\mathbf{b} = (b_{\mathrm{ew}},\, b_{\mathrm{ns}},\, 0)$ (east, north, up). The unit vector toward the source in equatorial coordinates is $\hat{s} = (\cos\delta\cos h,\;\cos\delta\sin h,\;\sin\delta)$. Projecting $\hat{s}$ into the topocentric frame via the rotation matrix $R(\text{eq}\to\text{topo})$ that accounts for the observatory latitude $L$, the delay is $\tau_g = \mathbf{b}\cdot\hat{s}_{\mathrm{topo}}/c$. Carrying out the matrix multiplication yields the expression above; see [TMS3] §4.1 for the explicit rotation.

The north–south contribution has two terms: a $\cos h$ term (time-varying) and a constant term $-(b_{\mathrm{ns}}/c)\cos L\,\sin\delta$ that is absorbed into an effective cable delay:

$$\tau'_c = \tau_c - \frac{b_{\mathrm{ns}}}{c}\cos L\,\sin\delta.$$


## 1.2 The Fringe Pattern (Point Source)

The NCH interferometer is an **adding** interferometer (after [Ryle52]): it combines the voltages from the two antennas, passes the sum through a square-law (power) detector, and time-averages to suppress the radio-frequency oscillation. The detected power is

$$P(t) = \bigl[E_1(t) + E_2(t)\bigr]^2 = E_0^2\cos^2(2\pi\nu t) + 2E_0^2\cos(2\pi\nu t)\cos\!\bigl[2\pi\nu(t+\tau_{\mathrm{tot}})\bigr] + E_0^2\cos^2\!\bigl[2\pi\nu(t+\tau_{\mathrm{tot}})\bigr].$$

Expanding the cross-term with the product-to-sum identity and time-averaging to remove all terms oscillating at $2\nu$:

$$\langle P \rangle = \underbrace{E_0^2}_{\text{DC (total power)}} + \underbrace{E_0^2\cos\!\bigl[2\pi\nu\,\tau_{\mathrm{tot}}(h)\bigr]}_{\text{fringe}}.$$

The first term is the sum of the two individual antenna powers — a slowly-varying DC baseline that is removed in post-processing (Section 2). The oscillating cross-term is the **fringe**, carrying all geometric information. After DC removal:

$$F(h) = E_0^2 \cos\!\bigl[2\pi\nu\,\tau_{\mathrm{tot}}(h)\bigr].$$

> **Note (correlating interferometer).** A correlating interferometer computes $\langle E_1 E_2\rangle$ directly, yielding $\tfrac{E_0^2}{2}\cos(2\pi\nu\tau_{\mathrm{tot}})$ — the same fringe phase and period, without a DC offset and with half the amplitude. Both architectures encode identical geometric information; the adding interferometer is simpler to build at the cost of requiring DC removal. See [TMS3] §1.3 for the historical comparison and [Ryle52] for the original adding-interferometer paper.

Expanding the total delay $\tau_{\mathrm{tot}} = \tau'_g(h) + \tau'_c$ and absorbing the (unknown) cable-delay phase $2\pi\nu\tau'_c$ into two fitting constants $A$ and $B$:

$$\boxed{F(h) = A\cos\!\bigl(2\pi\nu\,\tau'_g(h)\bigr) + B\sin\!\bigl(2\pi\nu\,\tau'_g(h)\bigr),}$$

where $\tau'_g(h) = (b_{\mathrm{ew}}/c)\cos\delta\,\sin h + (b_{\mathrm{ns}}/c)\sin L\,\cos\delta\,\cos h$ is the time-varying part of the geometric delay (the effective geometric delay). The constants satisfy $A = E_0^2\cos(2\pi\nu\tau'_c)$ and $B = -E_0^2\sin(2\pi\nu\tau'_c)$, so the total amplitude is $\sqrt{A^2 + B^2} = E_0^2$ and the phase offset is $\phi_0 = \arctan(-B/A) = 2\pi\nu\tau'_c$.

This is the fundamental equation of the adding interferometer: for a point source, the fringe is a quasi-sinusoidal function of hour angle whose argument depends on the baseline components and source declination.

> **Caveats — what is hidden in $A$, $B$, and $\tau'_c$.** The treatment above assumes a single, time-invariant cable phase and a single per-antenna voltage gain $E_0$ that is identical for both antennas and constant across the analysis band. In practice each receiver has its own complex gain $g_i(\nu, t) = |g_i(\nu)|\,e^{i\phi_i(\nu, t)}$ that drifts with temperature, has frequency structure (bandpass), and may shift between observing "chips" (segments separated by retunes or restarts of the back-end). The boxed equation should therefore be read as the **per-channel, per-chip** model: $A$ and $B$ are local fitting constants, not global ones, and any analysis that pools data across chips must either solve for, or remove, the chip-to-chip gain and phase offsets first. ([TMS3] §10 and §11 discuss the standard CLEAN/self-calibration approaches that lift these restrictions for production interferometers.)
>
> **Bandwidth and time decorrelation.** Two further effects are small but not zero. (i) *Bandwidth decorrelation* multiplies $F$ by $\mathrm{sinc}(\pi\,\Delta\nu\,\tau'_g)$, where $\Delta\nu$ is the channel width ([TMS3] §6.3, eq. 6.61). With $\Delta\nu = F_S/N_{\mathrm{FFT}} = 244\;\mathrm{kHz}$ and $|\tau'_g| \lesssim 50\;\mathrm{ns}$ (set by the maximum delay $b_{\mathrm{ew}}/c \approx 50\;\mathrm{ns}$), the argument $\pi\,\Delta\nu\,\tau'_g \lesssim 0.038$, giving a per-channel loss of $\sim (\pi\Delta\nu\tau)^2/6 \approx 2.4\times 10^{-4}$ — completely ignorable. The envelope tilt across the 70 MHz analysis band, $\sim (\pi\,\Delta\nu_{\mathrm{band}}\tau)^2/6$ with $\Delta\nu_{\mathrm{band}} = 70\;\mathrm{MHz}$, is $\sim 10\%$ at the longest delays — *not* negligible at the band-averaged level if pooled across many captures, which is why we work per-channel. (ii) *Time-average smearing*: integrating for $\Delta t$ multiplies by $\mathrm{sinc}(\pi f_f\,\Delta t)$. With $\Delta t \approx 2.5\;\mathrm{s}$ and $|f_f| \lesssim 0.05\;\mathrm{Hz}$ at transit, the argument $\pi\,f_f\,\Delta t \lesssim 0.39$, giving $\mathrm{sinc} \approx 0.974$, i.e. a loss of $\sim 2.5\,\%$ at transit and smaller toward the horizon — small enough to fold into the noise budget but worth flagging.


## 1.3 Fringe Frequency

The fringe phase $\phi(h) = 2\pi\nu\,\tau'_g(h)$ changes as the Earth rotates the source through the fringe pattern. The instantaneous fringe frequency (in Hz) is

$$f_f = \frac{1}{2\pi}\frac{d\phi}{dt} = \nu\,\frac{d\tau'_g}{dt}.$$

Since $h$ changes at the sidereal rate $\omega_\oplus = 2\pi / T_{\mathrm{sid}} \approx 7.292115\times10^{-5}\;\mathrm{rad\,s^{-1}}$ (with $T_{\mathrm{sid}} = 86\,164.0905\;\mathrm{s}$, [IERS2010]), differentiating $\tau'_g$ with respect to $h$ and multiplying by $\omega_\oplus$:

$$\boxed{f_f(h) = \omega_\oplus\left[\frac{b_{\mathrm{ew}}}{\lambda}\cos\delta\,\cos h - \frac{b_{\mathrm{ns}}}{\lambda}\sin L\,\cos\delta\,\sin h\right],}$$

where $\lambda = c/\nu$ is the observing wavelength (with $c = 299\,792\,458\;\mathrm{m\,s^{-1}}$ defined exactly by the SI, [BIPM-SI]).

### Fringe period at the meridian

At transit ($h = 0$) with $b_{\mathrm{ns}} \approx 0$:

$$P_f = \frac{1}{|f_f|} = \frac{\lambda}{\omega_\oplus\, b_{\mathrm{ew}}\,\cos\delta} \approx \frac{26\;\mathrm{s}}{\cos\delta}$$

for $b_{\mathrm{ew}} \approx 15.17\;\mathrm{m}$ (lidar prior, see `constants.py`) and $\lambda \approx 2.87\;\mathrm{cm}$ at $10.45\;\mathrm{GHz}$ ($P_f = 0.02868\,\mathrm{m} / (7.292\times10^{-5}\,\mathrm{rad\,s^{-1}} \times 15.17\,\mathrm{m}) \approx 25.9\;\mathrm{s}$). The fringe period is **shortest at transit** (where $|\cos h| = 1$) and stretches toward infinity at the horizon ($|\cos h| \to 0$).

### The x-coordinate linearisation

The fringe phase is $\phi = 2\pi(b_{\mathrm{ew}}/\lambda)\,x$ where $x \equiv \cos\delta\,\sin h$. In this coordinate the fringe is a **pure sinusoid**, which means an FFT in $x$-space directly yields the spatial frequency $b_{\mathrm{ew}}/\lambda$. This is the basis of the FFT baseline method (Section 3.1; the same trick is described in [TMS3] §10.4 in the context of single-baseline calibration).

### Delay vs sky-plane baseline — a critical distinction

Two different projections of the baseline arise naturally:

| Quantity | Formula (EW baseline, $b_{\mathrm{ns}} = 0$) | At transit | At horizon |
|----------|-------|------------|------------|
| **Delay baseline** $w = \nu\tau_g$ | $(b_{\mathrm{ew}}/\lambda)\cos\delta\,\sin h$ | **0** | **max** |
| **Sky-plane baseline** $\|u_{\mathrm{sky}}\|$ | $(b_{\mathrm{ew}}/\lambda)\cos\delta\,\|\cos h\|$ (small-$\delta$ approx.) | **max** | **0** |

These are **complementary in the equatorial plane**: $w^2 + u_{\mathrm{sky}}^2 = (b_{\mathrm{ew}}\cos\delta/\lambda)^2$ when $\sin\delta = 0$.

> **More carefully.** The exact relation in $(u, v, w)$ ([TMS3] eq. 4.5) is $u^2 + v^2 + w^2 = |b|^2/\lambda^2$. For a pure EW baseline at hour angle $h$, declination $\delta$, the standard expressions are
> $$u = (b_{\mathrm{ew}}/\lambda)\cos h, \quad v = (b_{\mathrm{ew}}/\lambda)\sin\delta\,\sin h, \quad w = -(b_{\mathrm{ew}}/\lambda)\cos\delta\,\sin h,$$
> so the **true** sky-plane magnitude is $\sqrt{u^2+v^2} = (b_{\mathrm{ew}}/\lambda)\sqrt{\cos^2 h + \sin^2\delta\,\sin^2 h}$. The simpler form $|u_{\mathrm{sky}}| = (b_{\mathrm{ew}}/\lambda)\cos\delta\,|\cos h|$ used in the table is the **small-$\delta$ approximation** valid when $\sin^2\delta \ll 1$. For the Sun in the present dataset $\delta \approx -0.08^\circ$, so the approximation is exact at the $10^{-6}$ level and is what is used in the implementation. The table identity $w^2 + u_{\mathrm{sky}}^2 = (b_{\mathrm{ew}}\cos\delta/\lambda)^2$ should therefore be read as a small-$\delta$ statement, not a general identity.

- The **delay** $w$ determines the fringe *phase* (path difference between antennas). It is zero at transit because the source is perpendicular to the baseline.
- The **sky-plane baseline** $|u_{\mathrm{sky}}|$ determines the *spatial resolution* — how many fringe cycles fit across the source. It is **maximum at transit** (the full baseline is projected onto the sky) and zero at the horizon (the baseline is parallel to the line of sight).

The Bessel-function modulation of an extended source (Section 1.5) depends on $|u_{\mathrm{sky}}|$, **not** on $w$.


## 1.4 Extended Sources and the Van Cittert–Zernike Theorem

For an extended source with one-dimensional brightness distribution $I(\theta)$ (brightness as a function of angular offset $\theta$ in radians from the phase centre), the interferometer response is the **convolution** of the point-source fringe pattern with the source brightness. In the visibility domain this becomes a multiplication:

$$R(h) = F(h) \times \underbrace{\int_{-\infty}^{\infty} I(\theta)\,e^{-2\pi i\,|u_{\mathrm{sky}}|\,\theta}\,d\theta}_{\displaystyle V(|u_{\mathrm{sky}}|)\;\text{(complex visibility)}}.$$

This is the one-dimensional form of the **van Cittert–Zernike theorem** ([vC34], [Z38]; for the radio-astronomical statement and the extension to two dimensions and partially polarised sources see [TMS3] Ch. 14, eq. 14.7).

For a source brightness that is **real and symmetric** about the phase centre — as for a centred uniform disk — the imaginary part of the integral vanishes and the visibility reduces to a Fourier *cosine* transform,

$$V(|u_{\mathrm{sky}}|) \;=\; \int_{-\infty}^{\infty} I(\theta)\,\cos\!\bigl(2\pi\,|u_{\mathrm{sky}}|\,\theta\bigr)\,d\theta \;\equiv\; \mathrm{MF}(|u_{\mathrm{sky}}|).$$

The **modulating factor** $\mathrm{MF}(|u_{\mathrm{sky}}|)$ is therefore the Fourier transform of the source brightness, evaluated at the instantaneous sky-plane baseline $|u_{\mathrm{sky}}|$ (in wavelengths). The argument $2\pi|u_{\mathrm{sky}}|\,\theta$ is dimensionless (wavelengths $\times$ radians), as required. Since $|u_{\mathrm{sky}}| = |f_f|/\omega_\oplus$ (see Section 1.3), this is equivalent to taking the Fourier transform at fringe frequency $f_f$ with $\theta$ expressed in time units via $\theta_{\mathrm{time}} = \theta/\omega_\oplus$.

> **Why we keep the cosine form for the disk and the full complex form for the spot.** When the source brightness is *not* symmetric about the phase centre — for example, the disk plus an off-axis sunspot — the imaginary part of the integral does *not* vanish, and the modulating factor becomes complex. A point source displaced by $\Delta\alpha$ contributes a visibility $\propto e^{-2\pi i\,u\,\Delta\alpha}$ — a pure phase whose slope encodes the offset (used in §1.7 to localise sunspots). For the centred uniform disk treated in §1.5 the cosine form is exact; for §1.7 we keep the full complex form.

**Physical intuition:** when the fringe spacing $\lambda/|u_{\mathrm{sky}}|$ is much larger than the source, fringe peaks and troughs cover the source uniformly and the response is just the point-source fringe scaled by total flux. When the fringe spacing equals the source size, equal amounts of the source fall on positive and negative fringe lobes, and the response **cancels** — producing a null in the fringe amplitude envelope.


## 1.5 The Uniform Disk: Bessel-Function Visibility

### Brightness distribution

Model the Sun as a uniformly bright circular disk of angular radius $R$. For a 1-D cut through the disk centre, the chord length at offset $\theta$ is proportional to $\sqrt{R^2 - \theta^2}$, giving the projected brightness profile:

$$I(\theta) = \begin{cases} \displaystyle\frac{2}{\pi R^2}\sqrt{R^2 - \theta^2} & |\theta| < R \\[6pt] 0 & \text{otherwise}. \end{cases}$$

This is normalised to unit total flux: $\int_{-R}^{R} I(\theta)\,d\theta = 1$. Equivalently it is the projection along $v$ of a uniform 2-D disk, and by the Fourier slice theorem its 1-D Fourier transform is the same jinc as the 2-D Hankel transform of the disk ([B&W] §8.5, eq. 8.5.20).

### Visibility function

The Fourier transform of a uniform disk is a standard result in diffraction theory ([B&W] §8.5; [TMS3] §13.1, eq. 13.10). Substituting $I(\theta)$ into the MF integral (Section 1.4) and using the identity $\int_{-R}^{R}\sqrt{R^2-\theta^2}\,e^{-2\pi i u\theta}\,d\theta = \pi R^2 J_1(2\pi Ru)/(2\pi Ru)$, the normalised complex visibility is

$$\boxed{\frac{V(|u_{\mathrm{sky}}|)}{V(0)} = \frac{2\,J_1(2\pi\,|u_{\mathrm{sky}}|\, R)}{2\pi\,|u_{\mathrm{sky}}|\, R},}$$

where $J_1$ is the Bessel function of the first kind of order one ([DLMF] §10.2; [A&S] §9.1), and

$$|u_{\mathrm{sky}}| = \frac{|f_f|}{\omega_\oplus} = \left|\frac{b_{\mathrm{ew}}}{\lambda}\cos\delta\,\cos h - \frac{b_{\mathrm{ns}}}{\lambda}\sin L\,\cos\delta\,\sin h\right|$$

is the component of the baseline **perpendicular to the line of sight**, in units of wavelengths. This is the "jinc" function — the circular analogue of the sinc function.

**1-D projection approximation.** Strictly, the 2-D visibility of a circular disk is $2J_1(2\pi R\sqrt{u^2+v^2})/(2\pi R\sqrt{u^2+v^2})$, where $(u, v)$ are the full sky-plane baseline components (EW and NS). Here we use only $|u_{\mathrm{sky}}| \approx |u|$, dropping $v$. From §1.3, for a pure EW baseline $v = (b_{\mathrm{ew}}/\lambda)\sin\delta\sin h$. With $\delta \approx -0.08^\circ$ for the Sun in this dataset, $|v|/|u| < 0.0014\,|\tan h|$, i.e. $\lesssim 10^{-3}$ even at $|h| = 80^\circ$. The fractional bias on the inferred diameter is then $\lesssim (v/u)^2/2 \sim 10^{-6}$ — completely negligible. For an observation at a more inclined declination (e.g. a planet), this approximation would have to be revisited.

### Limb brightening at X-band — and why we still use a uniform disk

The Sun's optical photospheric diameter at 1 AU has a mean value of $\sim 31.99'$ from helioseismic ([BCD98]) and direct-imaging ([Meftah18]) determinations; Earth's orbital eccentricity ($\sim 1.7\%$) modulates the apparent diameter from $\sim 31.46'$ at aphelion (early July) to $\sim 32.53'$ at perihelion (early January). The value $\theta_\odot^{\mathrm{opt}} \approx 31.6'$ used in `constants.py` (`SOLAR_DIAMETER_ARCMIN_NOMINAL = 31.6`) is the operational value from the AY 121 lab manual ([AY121-Lab3]), appropriate for an observation near aphelion. The student should verify against the *current* lab manual and the actual observation date. At centimetre wavelengths the situation is different: the dominant emission mechanism is *thermal free–free* in the chromosphere and low corona ([Dulk85] §III; [R&L] §5.2 for the underlying Bremsstrahlung treatment), whose temperature *increases* with height according to the semi-empirical chromospheric model of [VAL81] (and updates [FAL93], [FAL2009]):

$$T_e \approx 6\times 10^3\;\mathrm{K}\;\text{(photosphere)} \;\to\; \approx 2\times 10^4\;\mathrm{K}\;\text{(top of chromosphere)} \;\to\; \approx 10^6\;\mathrm{K}\;\text{(corona)}.$$

At 10 GHz the optical depth $\tau_\nu = 1$ surface lies a few thousand km above the photosphere, with brightness temperature $T_b \approx (1\text{–}2)\times 10^4\;\mathrm{K}$ ([Zirin91]; [Stix] §10.3), and the *limb* is mildly **brighter** than the disk centre because the slant path through the chromosphere is longer ([Dulk85] §III.B; [Selhorst-modelling] for explicit modelling at 10 GHz; [Alissandrakis-ALMA] for ALMA-era cm/mm observations). The net effect is twofold:

1. The **apparent radio radius** is larger than the optical radius by a few percent (textbook discussion: [Stix] §10.3, [Dulk85] §III). The closest *measured* published radio radius to our 10 GHz observation is the Nobeyama Radioheliograph value of [Selhorst04] at 17 GHz: $$R_\odot(17\,\mathrm{GHz}) = 976.6 \pm 1.5'' \quad\Longleftrightarrow\quad \theta_\odot(17\,\mathrm{GHz}) = 32.55 \pm 0.05'$$ (NoRH average over 1992–2003). The K-band measurements of [Marongiu24] at 18.3 / 25.8 GHz give $R_\odot \approx 976\text{–}982''$, consistent with this picture. Our observation is at 10 GHz, slightly *lower* in frequency than the Selhorst+04 anchor, so the radius is expected to be *slightly larger* than $976.6''$ — longer wavelength probes a higher chromospheric layer. We therefore use $\theta_\odot^{\mathrm{radio}}(10\,\mathrm{GHz}) \approx 32.6'$ as the comparison value, with the understanding that this is an extrapolation from a 17 GHz measurement and the lab is, in fact, *measuring* the radius at 10 GHz directly.
2. The **brightness profile is not flat**: cm-wavelength models of the quiet Sun show modest limb brightening over the outer few percent of the disk radius — see the Selhorst-Costa modelling series and the Alissandrakis ALMA papers for representative profiles. Replacing the uniform disk by a limb-brightened disk shifts the *position of the first Bessel null* inward (i.e. apparent diameter outward) by an amount that depends on the assumed profile but is typically of order $0.5\text{–}1\,\%$ for the shapes published in that literature. **The 10–25 % limb-brightening number I used in an earlier draft of this notebook should be treated as a rough textbook range, not a quote from a specific measurement** — substitute the actual published profile when writing up.

We retain the uniform-disk model in this lab because (i) the fit has only one nonlinear parameter, (ii) limb brightening is smaller than our other systematics (chip-to-chip gain, baseline cm-uncertainty), and (iii) it is the standard textbook reference. **The implication is that the diameter we measure is intrinsically larger than $31.6'$ by a quantifiable amount, and any final number should be compared to the published radio value, not the optical one.** This bias is folded into the systematic error budget in notebook 05.

### Geometric intuition

At **transit** ($h = 0$), the EW baseline is perpendicular to the line of sight, so the full baseline is projected onto the sky: $|u_{\mathrm{sky}}|$ is **maximum**. Many fringe cycles span the solar disk, positive and negative contributions largely cancel, and the visibility is **low** (deep in the Bessel sidelobes).

At **sunrise/sunset** ($|h| \to 90°$), the baseline is nearly parallel to the line of sight, so $|u_{\mathrm{sky}}| \to 0$. The fringe spacing is much larger than the Sun, the entire disk contributes coherently, and the visibility is **maximum** ($\mathrm{jinc}(0) = 1$).

### Properties of the jinc function

Bessel-function zero values from [DLMF] Table 10.21.1 / [A&S] Table 9.5:

| Property | Value |
|----------|-------|
| $V(0)/V(0)$ | 1 (at horizon, where $\|u_{\mathrm{sky}}\| \to 0$) |
| First null | $2\pi\,\|u_{\mathrm{sky}}\|\,R = j_{1,1} \approx 3.8317$ |
| Second null | $2\pi\,\|u_{\mathrm{sky}}\|\,R = j_{1,2} \approx 7.0156$ |
| Third null | $2\pi\,\|u_{\mathrm{sky}}\|\,R = j_{1,3} \approx 10.1735$ |

The **fringe amplitude envelope** traces the Bessel function from the main lobe (at the horizon) inward through successive nulls and sidelobes toward transit. For $b_{\mathrm{ew}} = 15.17\;\mathrm{m}$ and $R \approx 16'$, the maximum sky-plane baseline at transit is $|u_{\mathrm{sky}}|_{\max} \approx 530\;\lambda$, giving a Bessel argument $2\pi |u| R \approx 15.5$ — between the **fourth and fifth Bessel zeros**. So the data should show four full Bessel oscillations (and four nulls) between sunrise and transit.


## 1.6 Measuring the Solar Diameter

From the Bessel-function visibility, the $k$-th null occurs when

$$|u_{\mathrm{sky},k}|\,R = \frac{j_{1,k}}{2\pi}.$$

Given an observed null at sky-plane baseline $|u_{\mathrm{sky},k}|$ (in wavelengths), the angular radius is

$$\boxed{R = \frac{j_{1,k}}{2\pi\,|u_{\mathrm{sky},k}|}.}$$

(Bessel-zero values from [DLMF] §10.21 / [A&S] Table 9.5; the same diameter-from-null formula appears in the early radio-Sun observations of [Christiansen & Warburton 1955, Aust. J. Phys. 8, 474] and in the textbook treatment [TMS3] §13.1.)

Since $|u_{\mathrm{sky}}| \propto |\cos h|$, each null corresponds to a specific hour angle $h_k$ where $|\cos h_k|$ takes the appropriate value. Multiple nulls provide independent estimates whose scatter is a first-pass error bar.

A more precise estimate comes from a **nonlinear least-squares fit of the full Bessel envelope**, which uses every data point — not just the nulls — and gives a covariance matrix from the residuals. We report both. The formal covariance error is multiplied by $\sqrt{\chi^2_\nu}$ when $\chi^2_\nu > 1$, so that unmodelled systematics inflating the residuals are absorbed into the quoted uncertainty rather than ignored. The dominant terms in the diameter error budget are then:

| Source | Approx. fractional contribution | Reference |
|---|---|---|
| Statistical (envelope SNR + fit residuals) | $\sim 0.1\text{–}0.2\,\%$ | radiometer eq., [R&W] §4 / [TMS3] §6.2 |
| Baseline uncertainty $\sigma_{b}/b \sim 0.30/15.17$ | $\sim 2.0\,\%$ | lidar prior, `constants.py` |
| Limb brightening (uniform-disk bias) | $\sim 0.5\text{–}1\,\%$ | [Selhorst-modelling], [Alissandrakis-ALMA] |
| Bandwidth smearing of $|u|$ across analysis band | $\sim 0.3\,\%$ | [TMS3] §6.3 |
| Chip-to-chip gain step | $\lesssim 0.5\,\%$ | empirical, see nb 02a |

added in quadrature these give a realistic ~1.4 % total — i.e. ~0.45' on a ~32' diameter, with the lidar-baseline and limb-brightening systematics contributing comparably.

## 1.7 Sunspot Signatures in the Visibility

A sunspot is a localised region of enhanced (or, more commonly in radio, *suppressed*) emission offset from the disk centre by angular displacement $(\Delta\alpha,\,\Delta\delta)$. The radio behaviour of sunspots is mode-dependent: at 10 GHz the dominant emission above strong sunspot magnetic fields is **gyroresonance** at the second/third harmonic of the local cyclotron frequency $\nu_{\rm gyro} = eB/(2\pi m_e c)$, which can give either bright or dark spots depending on the line-of-sight magnetic geometry ([Dulk85] §IV; [BBG98] §3). The lab does not need to identify which mechanism is at work — only that the spot contributes a localised flux to the visibility.

By the linearity of the van Cittert–Zernike theorem ([TMS3] §14.1), the total visibility of disk + spot is

$$V_{\mathrm{total}} = (1 - f)\,V_{\mathrm{disk}} + f\,V_{\mathrm{spot}},$$

where $f$ is the fractional flux of the spot and $V_{\mathrm{spot}} = V_{\mathrm{pt}}\,e^{-2\pi i\,(u\,\Delta\alpha + v\,\Delta\delta)}$ is a point-source visibility with an additional phase from the spot's angular offset. The spot is treated as **unresolved** ($V_{\mathrm{pt}} \equiv 1$) — for a single 15.17 m baseline at $\lambda \approx 2.87\;\mathrm{cm}$, the fringe spacing is $\lambda/b_{\mathrm{ew}} \approx 6.5'$ (so the effective angular resolution is roughly half that, $\sim 3'$), and individual sunspot umbrae ($\sim 0.5'$, see [Stix] §8.2) are well below this. (A two-element interferometer does not strictly have a *synthesised* beam in the imaging sense — that would require multiple baselines — so the right comparison is fringe spacing vs feature size.)

**Key diagnostic — amplitude.** At the Bessel nulls of the disk, $V_{\mathrm{disk}} = 0$, so $|V_{\mathrm{total}}| \approx f$. A non-zero fringe amplitude at what should be a null is direct evidence of asymmetric structure — the **residual amplitude at the null directly measures the spot's flux fraction**:

$$f \approx \frac{|V_{\mathrm{null}}|}{|V(0)|}.$$

Caveats: (i) the *measured* envelope minimum does not sit exactly at the theoretical Bessel zero (because of finite cadence and the rapidly-changing $u$ near the null), so a small residual remains even with no spot; (ii) the formula assumes a single spot, whereas multiple spots add coherently with phase factors that can constructively or destructively interfere; (iii) limb brightening of the *disk* itself contributes a residual of similar magnitude — the apparent "$f$" is contaminated at the few-percent level (see §8.1 of nb 05 for the budget).

**Key diagnostic — phase (added in this lab).** At a disk null, $V_{\mathrm{disk}} \approx 0$ so $V_{\mathrm{total}}(u) \approx f\,e^{-2\pi i\,u\,\Delta\alpha}$ (with $v\Delta\delta$ negligible since $v \approx 0$ for this dataset). The **slope of $\arg(V_{\mathrm{total}})$ vs $u$** in a small window around the null therefore directly gives the **EW offset** of the spot from disk centre:

$$\Delta\alpha = -\frac{1}{2\pi}\,\frac{d\arg(V_{\mathrm{total}})}{du}.$$

This is the standard Fourier-imaging interpretation of a point source displaced from the phase centre ([TMS3] §3.1, eq. 3.7).

The NS offset $\Delta\delta$ is **unconstrained** by a single-baseline EW interferometer — it would require either an NS baseline component large enough to give significant $v$, or measurements at multiple parallactic angles (the standard "rotation-synthesis" technique of [Ryle 1962, Nature 194, 517]; see [TMS3] §1.3 for the historical sketch). We therefore report only the EW offset.

Both diagnostics are implemented in `utils/solar_analysis.py::detect_sunspot_anomalies` (amplitude) and `localize_sunspot_phase` (joint amplitude + phase fit returning $f$ and $\Delta\alpha$).

---


## 1.7b Noise model and per-capture uncertainty

All five baseline-fitting methods (and the diameter / spot fits) need a per-capture uncertainty $\sigma$ on the complex visibility. We use two complementary estimates and take the larger:

1. **Per-channel scatter inside the analysis band.** For each capture $i$, the standard deviation of $|V_i(\nu)|$ across the $\sim 285$ good channels in the 70 MHz analysis band, divided by $\sqrt{N_{\mathrm{ch}}}$, gives an i.i.d.-channel estimate of the band-averaged $\sigma$. The correlator records this per-channel uncertainty in `corr_std`; we cross-check the two and they agree to factors of order unity.
2. **Off-fringe residual scatter.** After subtracting the best-fit fringe model from the data, the standard deviation of the residuals — in a small window of contiguous captures — provides a *post-fit* estimate that is insensitive to channel-to-channel coherence and naturally folds in non-Gaussian systematics.

The radiometer expectation for an adding interferometer with system temperature $T_{\mathrm{sys}}$, channel width $\Delta\nu$, and integration time $\Delta t$ is

$$\sigma_V \;\sim\; \frac{T_{\mathrm{sys}}}{\sqrt{\Delta\nu\,\Delta t}},$$

(the standard radiometer equation; [R&W] §4.1, eq. 4.13; [TMS3] §6.2, eq. 6.50). A quick consistency check (with $T_{\mathrm{sys}} \sim 100\;\mathrm{K}$ — the typical NCH X-band system temperature documented in [AY121-Lab3] — $\Delta\nu = 244\;\mathrm{kHz}$, $\Delta t \approx 2.5\;\mathrm{s}$, source flux $T_b \sim 10^4\;\mathrm{K}$ from [Zirin91]) gives an SNR per channel per capture of order $10^2$, which matches the empirical scatter.

When envelope smoothing is applied (e.g. in `extract_fringe_envelope`), the per-point sigmas are *inflated* by $\sqrt{N_{\mathrm{smooth}}}$ to account for the loss of independence between neighbouring smoothed samples. Without this correction, $\chi^2$-based covariance estimates collapse to artificially small uncertainties — this was the cause of the previously-reported "$\pm 0.00$ arcmin" diameter uncertainty (now fixed in `fit_solar_diameter_bessel`).

---

> **A note on references.** Every numerical claim, every named theorem, and every textbook technique used in this notebook is attributed to an entry in [`labs/03/REFERENCES.md`](../REFERENCES.md). Short keys like `[TMS3]`, `[VAL81]`, `[Selhorst04]`, `[BCD98]`, `[DLMF]` resolve to full bibliographic entries there. Where I am genuinely unsure of a specific paper's volume/page (cm-radio Sun radius is the most common case), I cite generically with "see e.g. ... and references therein" rather than fabricating a citation.


## 1.8 Baseline Determination: Five Methods

The fringe phase (Section 1.2) encodes the baseline components $(b_{\mathrm{ew}}, b_{\mathrm{ns}})$. We now derive five independent methods to extract them from the observed complex visibility $V(h) = |V|\,e^{i\phi(h)}$. The first three are standard textbook techniques ([TMS3] §10–12 covers all of them in the framework of single-baseline calibration); the last two are direct nonlinear fits of the forward model.

All methods start from the geometric delay at capture $i$:

$$\tau'_{g,i} = \frac{b_{\mathrm{ew}}}{c}\cos\delta_i\,\sin h_i + \frac{b_{\mathrm{ns}}}{c}\sin L\,\cos\delta_i\,\cos h_i,$$

where $\delta_i$ is the per-capture source declination (which varies slowly as the Sun moves). The baselines $b_{\mathrm{ew}}$ and $b_{\mathrm{ns}}$ are constant — only the geometry changes.

---

### Method 1: FFT in the x-coordinate

**Observable:** Complex visibility $V(h)$.
**Key idea:** The coordinate $x_i \equiv \cos\delta_i\,\sin h_i$ linearises the fringe phase for a pure EW baseline. (The $x$-substitution trick is described in [TMS3] §10.4 for single-baseline geometry calibration.)

Since $\phi = 2\pi(b_{\mathrm{ew}}/\lambda)\,x + (\text{terms involving } \cos h) + \phi_c$, the visibility (for $b_{\mathrm{ns}} \approx 0$) is a pure sinusoid in $x$ with spatial frequency $f_x = b_{\mathrm{ew}}/\lambda$.

**Baseline recovery:**

$$\boxed{b_{\mathrm{ew}} = |f_{x,\mathrm{peak}}| \times \frac{c}{\nu}.}$$

**Limitation:** Measures only $b_{\mathrm{ew}}$; cannot recover $b_{\mathrm{ns}}$.

---

### Method 2: Phase slope

**Observable:** Unwrapped phase $\phi(h) = \arg V(h)$.
**Key idea:** Fit the per-capture phase using the physical regressors that encode the per-capture geometry. (Standard linear-regression interferometric calibration; [TMS3] §10.1.)

The design matrix is $\mathbf{X} = [\cos\delta_i\sin h_i,\;\sin L\,\cos\delta_i\cos h_i,\;\mathbf{1}]$, and the fit gives:

$$\phi_i = A\,\cos\delta_i\sin h_i + B\,\sin L\,\cos\delta_i\cos h_i + \phi_0,$$

where $A = 2\pi b_{\mathrm{ew}}/\lambda$ and $B = 2\pi b_{\mathrm{ns}}/\lambda$, so:

$$\boxed{b_{\mathrm{ew}} = \frac{A\,c}{2\pi\nu}, \qquad b_{\mathrm{ns}} = \frac{B\,c}{2\pi\nu}.}$$

Note: because the per-capture $\cos\delta_i$ is folded into the regressors, the baseline recovery is a simple scaling — no division by $\cos\delta$ is needed.

**Uncertainty:** $\mathrm{Cov}(\hat{\boldsymbol{\beta}}) = \hat{\sigma}^2\,(\mathbf{X}^T\mathbf{X})^{-1}$ ([Bevington & Robinson 2003] / [AY121-FitNotes]).

---

### Method 3: Lag-spectrum delay

**Observable:** Complex cross-spectrum $V(\nu, h)$ across frequency channels.
**Key idea:** IFFT gives the geometric delay $\tau_g$ per capture (this is the standard "delay calibration" approach used in connected-element arrays such as the VLA; [TMS3] §10.2); fit these delays using the same physical regressors:

$$\tau_i = b_{\mathrm{ew}}\,\frac{\cos\delta_i\,\sin h_i}{c} + b_{\mathrm{ns}}\,\frac{\sin L\,\cos\delta_i\,\cos h_i}{c} + \tau_{\mathrm{inst}}.$$

**Baseline recovery:** The fit coefficients directly give $b_{\mathrm{ew}}$ and $b_{\mathrm{ns}}$ (after unit conversion from nanoseconds to metres).

---

### Method 4: Nonlinear least-squares (NLS) complex fringe fit

**Observable:** Full complex visibility $V(h)$ (Re + Im).
**Key idea:** Fit the fringe equation directly, with $(b_{\mathrm{ew}}, b_{\mathrm{ns}})$ as nonlinear parameters and $(A, B, C, D)$ as linear parameters solved analytically at each step (the standard separable-variable trick of [Golub & Pereyra 1973, SIAM J. Numer. Anal. 10, 413]).

The model uses per-capture declination:

$$\psi_i = \frac{2\pi}{\lambda}\bigl[b_{\mathrm{ew}}\cos\delta_i\sin h_i + b_{\mathrm{ns}}\sin L\,\cos\delta_i\cos h_i\bigr],$$

$$V_i = (A + iC)\cos\psi_i + (B + iD)\sin\psi_i.$$

The optimizer searches over $(b_{\mathrm{ew}}, b_{\mathrm{ns}})$ in metres, seeded from the phase-slope result.

---

### Method 5: Brute-force grid search

**Observable:** Same as NLS (full complex visibility).
**Key idea:** Evaluate the NLS cost function on a 2-D grid over $(Q_{\mathrm{ew}}, Q_{\mathrm{ns}}) = (b_{\mathrm{ew}}/\lambda,\; b_{\mathrm{ns}}/\lambda)$, solving $(A, B, C, D)$ analytically at each grid point. Per-capture $\cos\delta_i$ enters the phase computation at every point.

Two passes: a **coarse** grid to locate the global minimum among the many sidelobes, then a **fine** grid to refine the position and compute the curvature matrix $[\alpha]$ and covariance $[\alpha]^{-1}$ for uncertainties (the Bevington-style $\chi^2$ procedure of [Bevington & Robinson 2003] §11; documented as the lab's standard fitting method in [AY121-FitNotes]).

---

### Summary

| Method | Observable | Recovers $b_{\mathrm{ns}}$? | Phase unwrapping? | Initial guess? | Per-capture $\delta$? | Reference |
|--------|-----------|:-:|:-:|:-:|:-:|---|
| FFT | Complex $V$ in $x$-space | No | No | No | Yes | [TMS3] §10.4 |
| Phase slope | $\arg(V)$ | Yes | Yes | No | Yes | [TMS3] §10.1 |
| Lag delay | $\|\mathrm{IFFT}(V(\nu))\|$ | Yes | No | No | Yes | [TMS3] §10.2 |
| NLS | Complex $V$ (Re + Im) | Yes | No | Yes (phase slope) | Yes | [Golub & Pereyra 1973] |
| Grid search | Complex $V$ (Re + Im) | Yes | No | No (exhaustive) | Yes | [Bevington & Robinson 2003] §11 |
